# otro_pipe/02 — descomposicion EEMD + pronostico por componente

Idea de la literatura de forecasting "decompose-ensemble": en vez de pronosticar `tn` de un
producto directo, se lo separa en sub-senales de distinta frecuencia con **EEMD** (Ensemble
Empirical Mode Decomposition), se pronostica cada sub-senal por separado, y se suman los
pronosticos para reconstruir `tn` (Wu, Huang & Chen, 2009; Lei & Zuo, 2009).

- EEMD agrega ruido blanco gaussiano a la serie, la descompone con EMD (Empirical Mode
  Decomposition) `m` veces con ruido distinto cada vez, y promedia las IMFs resultantes. Eso
  mitiga el *mode mixing* que tiene el EMD sin ensemble (una sola IMF mezclando frecuencias
  distintas).
- Se aplica sobre `tn` agregado por **producto** (no cliente-producto: es el nivel en el que se
  entrega el submit), producto por producto, con `m=100` corridas de ensemble y ruido de
  desvio `0.05` -- son los defaults del paper y tambien los defaults de la libreria.
- Cada producto se descompone en como mucho `eemd_max_imf` IMFs + 1 residuo. El residuo se
  define por resta (`residuo = serie - suma(IMFs)`), asi la reconstruccion es exacta sin
  importar cuantas IMFs haya encontrado el EMD (se probo que `sum(IMFs que devuelve la
  libreria) != serie original` -- por eso no alcanza con usar el residuo que devuelve PyEMD).
- Se entrena **un modelo LightGBM por componente** (una IMF = una frecuencia = un modelo, tal
  como describe la literatura), con lags/medias moviles de esa componente nada mas. El
  pronostico final es la suma de los pronosticos de todas las componentes.

Libreria: `EMD-signal` (se importa como `PyEMD`). No usa `duckdb`: a nivel producto el panel es
chico y alcanza con polars (mismo estilo que `pipe_demanda`).


## 0) Setup

In [ ]:
%pip install -q EMD-signal

In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
from PyEMD import EEMD
from tqdm.auto import tqdm


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_PRE  = BUCKET / "datasets_pre"
DIR_FE   = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp_otro_pipe"
for d in (DIR_PRE, DIR_FE, RUTA_EXP):
    d.mkdir(parents=True, exist_ok=True)

print(f"BUCKET   : {BUCKET}")
print(f"crudos   : {DIR_RAW}")
print(f"cache pre: {DIR_PRE}")
print(f"cache FE : {DIR_FE}")
print(f"salida   : {RUTA_EXP}")


In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


def a_m(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)


def m_a_periodo(m: int) -> int:
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


In [ ]:
PARAM = {
    'horizonte': 2,
    'max_lags': 12,
    'ventanas_ma': (3, 6),

    # -- EEMD (Wu, Huang & Chen, 2009) --------------------------------------
    # se descompone tn por PRODUCTO (nivel del submit), no cliente-producto.
    'eemd_trials': 100,          # m: corridas del ensemble (paper: 100)
    'eemd_noise_width': 0.05,    # sigma del ruido gaussiano (paper: 0.05 --
                                 # tambien el default de PyEMD, relativo al rango de la senal)
    'eemd_max_imf': 5,           # techo de IMFs por producto; el resto queda en el residuo
    'eemd_n_jobs': 1,            # >1 -> EEMD paraleliza los trials de ESE producto entre
                                 # procesos (0.17s/producto en serie con trials=100 y series
                                 # de ~36 meses -- no hace falta para volumenes tipicos)
    'min_meses_para_eemd': 18,   # series mas cortas: sin descomponer (todo al residuo) --
                                 # con pocos puntos el EMD no tiene extremos locales confiables
                                 # (y con 1 solo punto directamente tira excepcion)

    # -- Particion train/val/test (mismos cortes que el resto de la sesion) --
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # -- Entrega -------------------------------------------------------------
    'periodo_objetivo': 202002,
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,

    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
L = PARAM['max_lags']
COMPONENTES = [f"imf{k + 1}" for k in range(PARAM['eemd_max_imf'])] + ["residuo"]
N_COMP = len(COMPONENTES)

EXPERIMENTO = (f"eemd_trials{PARAM['eemd_trials']}_noise{PARAM['eemd_noise_width']}"
              f"_maximf{PARAM['eemd_max_imf']}_{L}lags_h{H}"
              + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"componentes ({N_COMP}): {COMPONENTES}")


## 1) Serie `tn` por producto, densificada con ceros dentro de su vida

In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")

tot_prod = (sell.group_by(["product_id", "periodo"])
                .agg(pl.col("tn").sum().alias("tn_prod"))
                .with_columns(a_m(pl.col("periodo")).alias("m")))

vida = tot_prod.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"))

# grilla SOLO dentro de la vida de cada producto (no antes de nacer, no despues de la
# ultima venta observada) -- ceros artificiales fuera de esa ventana meterian saltos que
# no existieron y le arruinarian la descomposicion al EMD.
grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").select("product_id", "m"))

tot_prod = (grilla.join(tot_prod.drop("periodo"), on=["product_id", "m"], how="left")
                  .with_columns(pl.col("tn_prod").fill_null(0.0))
                  .join(vida, on="product_id", how="left")
                  .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand"),
                        on="product_id", how="left")
                  .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64)
                                  .alias("periodo"))
                  .sort(["product_id", "m"]))

largo = vida.with_columns((pl.col("m_muere") - pl.col("m_nace") + 1).alias("n"))
print(f"productos: {tot_prod['product_id'].n_unique()}   filas: {tot_prod.height:,}   "
     f"[{time.time() - t0:.0f}s]")
print(f"largo de serie por producto: min {largo['n'].min()}  mediana {int(largo['n'].median())}  "
     f"max {largo['n'].max()}")


## 2) Descomposicion EEMD por producto (IMFs + residuo)

Se cachea en `datasets_fe/` igual que el resto de los pasos de FE del repo: si ya corrio una
vez con estos parametros, se saltea.


In [ ]:
_tag = f"eemd_trials{PARAM['eemd_trials']}_noise{PARAM['eemd_noise_width']}_maximf{PARAM['eemd_max_imf']}"
path_eemd = DIR_FE / f"{_tag}_componentes_otroPipe.parquet"

if path_eemd.exists():
    print(f"cache encontrada: {path_eemd.name} -> se salta la descomposicion")
    componentes_df = pl.read_parquet(path_eemd)
else:
    t0 = time.time()

    def descomponer(valores: np.ndarray) -> np.ndarray:
        """Devuelve un array (N_COMP, n) = [imf1..imfK, residuo]. El residuo se define
        por resta (residuo = valores - suma(IMFs)) -- la suma de las filas reconstruye
        'valores' exacto sin importar cuantas IMFs haya encontrado el EMD (probado que
        PyEMD NO garantiza que sum(IMFs devueltas) == serie original)."""
        n = len(valores)
        salida = np.zeros((N_COMP, n), dtype=np.float64)
        if n < PARAM['min_meses_para_eemd']:
            salida[-1] = valores  # serie corta: todo al residuo, sin descomponer
            return salida
        eemd = EEMD(trials=PARAM['eemd_trials'], noise_width=PARAM['eemd_noise_width'],
                    parallel=PARAM['eemd_n_jobs'] > 1,
                    processes=PARAM['eemd_n_jobs'] if PARAM['eemd_n_jobs'] > 1 else None)
        eemd.noise_seed(PARAM['semilla'])
        imfs = eemd.eemd(valores.astype(np.float64), max_imf=PARAM['eemd_max_imf'])
        k = min(imfs.shape[0], PARAM['eemd_max_imf'])
        salida[:k] = imfs[:k]
        salida[-1] = valores - salida[:-1].sum(axis=0)
        return salida

    filas = []
    n_prod = tot_prod['product_id'].n_unique()
    for (pid,), g in tqdm(tot_prod.sort(["product_id", "m"]).group_by("product_id", maintain_order=True),
                          total=n_prod, desc="EEMD por producto"):
        ms = g["m"].to_list()
        comp = descomponer(g["tn_prod"].to_numpy())
        for j, nombre in enumerate(COMPONENTES):
            filas.append(pl.DataFrame({"product_id": [pid] * len(ms), "m": ms,
                                       "componente": [nombre] * len(ms),
                                       "valor": comp[j].tolist()}))
    componentes_df = pl.concat(filas)
    componentes_df.write_parquet(path_eemd)
    print(f"descompuestos {n_prod} productos en {N_COMP} componentes cada uno.   "
         f"[{time.time() - t0:.0f}s]")
    print(f"Guardado: {path_eemd}")


In [ ]:
t0 = time.time()
chk_wide = componentes_df.pivot(on="componente", index=["product_id", "m"], values="valor")
chk_wide = chk_wide.with_columns(pl.sum_horizontal(COMPONENTES).alias("_suma"))
chk = chk_wide.join(tot_prod.select("product_id", "m", "tn_prod"), on=["product_id", "m"], how="inner")
error_max = float((chk["_suma"] - chk["tn_prod"]).abs().max())
assert error_max < 1e-6, f"la suma de componentes no reconstruye tn_prod (error max {error_max:.2e})"
print(f"chequeo round-trip: suma(IMFs + residuo) == tn_prod, error maximo {error_max:.2e}   "
     f"[{time.time() - t0:.0f}s]")


### Ejemplo visual (repasar a ojo antes de gastar tiempo entrenando)

In [ ]:
SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
TINTA, GRILLA_C = "#0b0b0b", "#e1e0d9"
plt.rcParams.update({"axes.grid": True, "grid.color": GRILLA_C, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9, "figure.dpi": 110})

top_vol = (tot_prod.group_by("product_id").agg(pl.col("tn_prod").sum().alias("t"))
                  .sort("t", descending=True).head(2)["product_id"].to_list())

for pid in top_vol:
    wide = (componentes_df.filter(pl.col("product_id") == pid)
                          .pivot(on="componente", index="m", values="valor").sort("m"))
    orig = tot_prod.filter(pl.col("product_id") == pid).sort("m")["tn_prod"]
    fig, axes = plt.subplots(N_COMP + 1, 1, figsize=(8, 1.3 * (N_COMP + 1)), sharex=True)
    axes[0].plot(wide["m"], orig, color=TINTA, linewidth=1.4)
    axes[0].set_title(f"producto {pid} -- tn_prod original", loc="left", fontsize=8)
    for j, nombre in enumerate(COMPONENTES, start=1):
        axes[j].plot(wide["m"], wide[nombre], color=SERIE[(j - 1) % len(SERIE)], linewidth=1.1)
        axes[j].set_title(nombre, loc="left", fontsize=8)
    fig.tight_layout()
    plt.show()


## 3) Features por componente (lags, medias moviles) + categorias de producto

In [ ]:
t0 = time.time()

panel = (componentes_df.join(tot_prod.select("product_id", "m", "m_nace", "m_muere",
                                             "cat1", "cat2", "cat3", "brand"),
                             on=["product_id", "m"], how="left")
                       .sort(["product_id", "componente", "m"]))

panel = panel.with_columns(
    *[pl.col("valor").shift(k).over(["product_id", "componente"]).alias(f"valor_lag{k}")
      for k in range(1, L + 1)],
    *[pl.col("valor").rolling_mean(w).over(["product_id", "componente"]).alias(f"valor_ma{w}")
      for w in PARAM['ventanas_ma']],
    (pl.col("m") - pl.col("m_nace")).alias("edad_producto"),
    ((pl.col("m") - 1) % 12 + 1).alias("mes_del_anio"),
    pl.col("valor").shift(-H).over(["product_id", "componente"]).alias("y"),
    (pl.col("m") + H).map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo_objetivo"),
)
panel = panel.with_columns(
    (pl.col("valor") - pl.col(f"valor_ma{PARAM['ventanas_ma'][0]}")).alias("valor_desvio_ma"))

CAT_FEATURES = ["cat1", "cat2", "cat3", "brand"]
panel = panel.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                            for c in CAT_FEATURES])

FEATURES = [c for c in panel.columns if c not in {
    "product_id", "componente", "m", "m_nace", "m_muere", "periodo_objetivo", "y",
}]
print(f"panel: {panel.height:,} filas x {len(FEATURES)} features   [{time.time() - t0:.0f}s]")
print(f"features: {FEATURES}")


## 4) Split train/val/test + control de leakage

In [ ]:
MESES_TRAIN = sorted(a_m(p) for p in PARAM['meses_train'])
MESES_VAL   = sorted(a_m(p) for p in PARAM['meses_val'])
MESES_TEST  = sorted(a_m(p) for p in PARAM['meses_test'])

sup = panel.filter(pl.col("y").is_not_null())
periodos_sup_m = set(sup["m"].unique().to_list())
MESES_TRAIN = [m for m in MESES_TRAIN if m in periodos_sup_m]
MESES_VAL   = [m for m in MESES_VAL if m in periodos_sup_m]
MESES_TEST  = [m for m in MESES_TEST if m in periodos_sup_m]
for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(f"{nombre} quedo vacio dentro de los periodos con target disponibles.")

MESES_INFER = sorted(panel.filter(pl.col("y").is_null())["m"].unique().to_list())[-H:]
infer = panel.filter(pl.col("m").is_in(MESES_INFER))

errores, avisos = [], []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE LEAKAGE")
print("=" * 72)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"), (MESES_VAL, MESES_TEST, "val", "test")):
    gap = min(b) - max(a)
    chk(gap >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es) >= horizonte {H}")
if PARAM['reentrenar_con_val_para_test']:
    gap_tv = min(MESES_TEST) - max(MESES_TRAIN + MESES_VAL)
    chk(gap_tv >= H, f"gap (train+val) -> test = {gap_tv} >= {H}")
chk(not (set(MESES_TRAIN) & set(MESES_VAL)), "train y val son disjuntos")
chk(not (set(MESES_VAL) & set(MESES_TEST)), "val y test son disjuntos")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST), "orden cronologico train < val < test")
chk(not (set(MESES_INFER) & set(MESES_TRAIN + MESES_VAL + MESES_TEST)),
   f"los meses de inferencia {MESES_INFER} no se usan para entrenar/validar/testear")
chk(not ({"y", "periodo_objetivo"} & set(FEATURES)), "el target no esta entre las features")

# la correlacion >0.999 con el target es sospechosa de leakage en el resto del repo, pero
# aca NO se trata como error duro: el residuo (tendencia suave) puede tener autocorrelacion
# legitimamente altisima a lag corto sin que eso sea un bug -- se deja como aviso.
y_chk = sup["y"].to_numpy().astype(np.float64)
for c in FEATURES:
    if c in CAT_FEATURES:
        continue
    x = sup[c].to_numpy().astype(np.float64)
    ok_mask = np.isfinite(x) & np.isfinite(y_chk)
    if ok_mask.sum() < 100 or x[ok_mask].std() == 0:
        continue
    r = float(np.corrcoef(x[ok_mask], y_chk[ok_mask])[0, 1])
    if abs(r) > 0.999:
        avisos.append((c, round(r, 5)))
if avisos:
    print(f"  [aviso] features con corr > 0.999 con y (revisar si tiene sentido, no es error "
         f"automatico por el residuo suave): {avisos}")

print("=" * 72)
if errores:
    raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
print(f"\nControl superado. TRAIN {len(MESES_TRAIN)} · VAL {len(MESES_VAL)} · TEST {len(MESES_TEST)} meses.")


## 5) WAPE + un modelo LightGBM por componente

In [ ]:
def wape(y_real, y_pred, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Identico al del resto de la sesion."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        _, inv = np.unique(np.asarray(product_ids), return_inverse=True)
        yr = np.bincount(inv, weights=yr)
        yp = np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


PARAMS_LGBM = {'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
              'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 31,
              'min_child_samples': 20, 'seed': PARAM['semilla'], 'n_jobs': -1,
              'deterministic': True, 'force_row_wise': True}
CAT_EN_FEATURES = [c for c in CAT_FEATURES if c in FEATURES]


def bloque(componente, meses):
    return sup.filter((pl.col("componente") == componente) & pl.col("m").is_in(meses))


def entrenar_componente(componente, meses_fit):
    b = bloque(componente, meses_fit).to_pandas()
    modelo = lgb.LGBMRegressor(**PARAMS_LGBM)
    modelo.fit(b[FEATURES], b["y"], categorical_feature=CAT_EN_FEATURES)
    return modelo


def predecir_componente(modelo, bloque_pl):
    return modelo.predict(bloque_pl.to_pandas()[FEATURES])


def evaluar(modelos, meses_ev):
    """Suma las predicciones de todas las componentes por (producto, mes) y compara
    contra el tn_prod real del mes objetivo (m + H)."""
    filas = []
    for comp in COMPONENTES:
        b = bloque(comp, meses_ev)
        if b.height == 0:
            continue
        pred = predecir_componente(modelos[comp], b)
        filas.append(b.select("product_id", "m").with_columns(pl.Series("pred", pred)))
    if not filas:
        return float("nan"), None
    pred_total = (pl.concat(filas).group_by(["product_id", "m"])
                        .agg(pl.col("pred").sum().alias("tn_pred"))
                        .with_columns((pl.col("m") + H).alias("m_obj")))
    real = tot_prod.select("product_id", pl.col("m").alias("m_obj"), "tn_prod")
    comparado = pred_total.join(real, on=["product_id", "m_obj"], how="inner")
    score = wape(comparado["tn_prod"].to_numpy(),
                np.maximum(comparado["tn_pred"].to_numpy(), PARAM['clip_min']),
                comparado["product_id"].to_numpy())
    return score, comparado


def wape_naive(meses_ev):
    """Baseline: pronostica tn_prod(t+H) = media movil de 3 meses de tn_prod en t."""
    real = tot_prod.select("product_id", pl.col("m").alias("m_obj"), "tn_prod")
    naive = (tot_prod.select("product_id", "m",
                             pl.col("tn_prod").rolling_mean(3).over("product_id").alias("tn_pred"))
                    .filter(pl.col("m").is_in(meses_ev))
                    .with_columns((pl.col("m") + H).alias("m_obj")))
    comparado = naive.join(real, on=["product_id", "m_obj"], how="inner")
    if comparado.height == 0:
        return float("nan")
    return wape(comparado["tn_prod"].to_numpy(), comparado["tn_pred"].fill_null(0.0).to_numpy(),
               comparado["product_id"].to_numpy())


print("motor de entrenamiento listo")


In [ ]:
t0 = time.time()
modelos_val = {comp: entrenar_componente(comp, MESES_TRAIN)
               for comp in tqdm(COMPONENTES, desc="entrenando (train->val)")}
wape_val, comparado_val = evaluar(modelos_val, MESES_VAL)
print(f"WAPE val (EEMD por componente): {wape_val:.5f}   [{time.time() - t0:.0f}s]")

t0 = time.time()
meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
modelos_test = {comp: entrenar_componente(comp, meses_fit_test)
                for comp in tqdm(COMPONENTES, desc="entrenando (->test)")}
wape_test, comparado_test = evaluar(modelos_test, MESES_TEST)
print(f"WAPE test (EEMD por componente): {wape_test:.5f}   [{time.time() - t0:.0f}s]")

naive_val, naive_test = wape_naive(MESES_VAL), wape_naive(MESES_TEST)
print(f"\nWAPE val:  EEMD {wape_val:.5f}   naive(ma3) {naive_val:.5f}")
print(f"WAPE test: EEMD {wape_test:.5f}   naive(ma3) {naive_test:.5f}")


## 6) Entrenamiento final + pronostico para `periodo_objetivo`

In [ ]:
t0 = time.time()
meses_fit_final = sorted(sup["m"].unique().to_list())
modelos_final = {comp: entrenar_componente(comp, meses_fit_final)
                 for comp in tqdm(COMPONENTES, desc="entrenando final")}

filas_pred = []
for comp in COMPONENTES:
    b = infer.filter(pl.col("componente") == comp)
    if b.height == 0:
        continue
    pred = predecir_componente(modelos_final[comp], b)
    filas_pred.append(b.select("product_id", "periodo_objetivo").with_columns(pl.Series("pred", pred)))
pred_final = (pl.concat(filas_pred).group_by(["product_id", "periodo_objetivo"])
                    .agg(pl.col("pred").sum().alias("tn")))

OBJ = PARAM['periodo_objetivo']
obj = pred_final.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_final['periodo_objetivo'].unique().to_list())}")

# productos oficiales sin prediccion (nunca vendieron o murieron mucho antes del corte) ->
# se entregan en 0, no hay de donde sacar una serie para descomponer.
submit = oficiales.select("product_id").join(obj.select("product_id", "tn"), on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0).clip(lower_bound=PARAM['clip_min'])).sort("product_id")
print(f"submit: {submit.height} filas, {sin_pred} sin prediccion (quedan en 0), "
     f"tn total {submit['tn'].sum():,.1f}   [{time.time() - t0:.0f}s]")


## 7) Entrega -- submission + `resultado.json` + Kaggle opcional

In [ ]:
path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
print(f"Guardado: {path_submit}")

resultado = {
    'experimento': EXPERIMENTO, 'periodo_objetivo': OBJ,
    'meses_train': PARAM['meses_train'], 'meses_val': PARAM['meses_val'], 'meses_test': PARAM['meses_test'],
    'wape_val': wape_val, 'wape_test': wape_test,
    'wape_naive_val': naive_val, 'wape_naive_test': naive_test,
    'eemd_trials': PARAM['eemd_trials'], 'eemd_noise_width': PARAM['eemd_noise_width'],
    'eemd_max_imf': PARAM['eemd_max_imf'], 'componentes': COMPONENTES,
    'sin_prediccion': sin_pred, 'tn_total_entregado': float(submit['tn'].sum()),
    'path_submit': str(path_submit.relative_to(BUCKET)),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / 'resultado.json', 'w', encoding='utf-8') as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
print(f"Guardado: {DIR_OUT / 'resultado.json'}")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. El CSV ya esta generado.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                print(f"Kaggle auth copiada de {cand}")
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube nada. El CSV ya esta generado.")
    else:
        kdst.chmod(0o600)
        msg = f"{EXPERIMENTO} | wape_test={wape_test:.5f}"
        ok, salida = kaggle_cli(["competitions", "submit", "-c", PARAM['kaggle_competition'],
                                "-f", str(path_submit), "-m", msg])
        print("Subida OK" if ok else f"Subida FALLO: {salida[-300:]}")


Repasar a ojo:
  - [ ] en los graficos de la seccion 2, la suma visual de IMFs + residuo, se parece a la
    serie original (`tn_prod`)?
  - [ ] el residuo (ultimo panel de cada grafico) se ve como una tendencia suave, sin los
    saltos mes a mes que si tiene `tn_prod`?
  - [ ] las IMFs de mayor frecuencia (`imf1`, `imf2`) oscilan alrededor de cero, sin
    arrastrar un nivel (si arrastran nivel, el residuo se esta quedando con parte de la
    tendencia que deberia ir en una IMF)?
  - [ ] el WAPE val/test del enfoque EEMD le gana al naive (`ma3`)? si no le gana, la
    descomposicion no esta aportando sobre simplemente seguir la tendencia reciente.
  - [ ] cuantos productos cayeron en `sin_prediccion` (fallback a 0)? tienen sentido
    comercial (productos discontinuados, sin ninguna venta historica)?
